# A complete study

One table carries the whole path: the sums, the quadratic, the comparison with the line, the prediction outside the sample, and a sampling check of the curvature's confidence interval. The algebra is repeated here so the notebook can be read on its own.


## The fit, once more

$$
\begin{array}{c|ccccc}
x & -2 & -1 & 0 & 1 & 2 \\
\hline
y & 10 & 0 & 1 & 8 & 16
\end{array}
$$

$$
\begin{aligned}
\sum y &= 35, & \sum xy &= 20, & \sum x^2 y &= 112, \\
10b_1 &= 20 &\Longrightarrow b_1 &= 2, \\
b_0 + 2b_2 &= 7, & 5b_0 + 17b_2 &= 56 &\Longrightarrow b_0 &= 1,\ b_2 = 3.
\end{aligned}
$$

Residuals $1, -2, 0, 2, -1$, so $\mathrm{RSS} = 10$ and $s^2 = 5$. The line $7 + 2x$ has $\mathrm{RSS} = 136$. The total sum of squares is $176$, and

$$
R^2 = \dfrac{166}{176} = \dfrac{83}{88}, \qquad
F = \dfrac{126/1}{10/2} = \dfrac{126}{5}
$$

for the hypothesis that the coefficient of $x^2$ is zero. At $x = 3$ the quadratic predicts $34$, a new observation has standard error $2\sqrt{7}$, and the cubic through these same points predicts $20$.


In [1]:
# Recompute the study numbers from the sums, including F and the two predictions at x=3.
from fractions import Fraction
import math
x = [-2, -1, 0, 1, 2]
y = [10, 0, 1, 8, 16]
b1 = Fraction(sum(xi * yi for xi, yi in zip(x, y)), 10)
b2 = Fraction(21, 7)
b0 = 7 - 2 * b2
rss = sum((yi - (b0 + b1 * xi + b2 * xi ** 2)) ** 2 for xi, yi in zip(x, y))
line_rss = 136
tss = 176
f_stat = Fraction(line_rss - rss, 1) / Fraction(rss, 2)
cubic_at_3 = 1 + Fraction(29, 6) * 3 + 3 * 9 - Fraction(5, 6) * 27
print("quadratic", b0, b1, b2, "RSS", rss, "F", f_stat)
print("prediction at 3", b0 + b1 * 3 + b2 * 9, "cubic", cubic_at_3)
print("prediction se", 2 * math.sqrt(7))
assert (b0, b1, b2, rss) == (1, 2, 3, 10)
assert f_stat == Fraction(126, 5)
assert 1 - Fraction(rss, tss) == Fraction(83, 88)
assert cubic_at_3 == 20


quadratic 1 2 3 RSS 10 F 126/5
prediction at 3 34 cubic 20
prediction se 5.291502622129181


## The sampling distribution of the curvature

Suppose the mean really is $1 + 2x + 3x^2$ at these five fixed inputs, and the errors are independent standard normal. The Gauss–Markov variance of $b_2$ is

$$
\operatorname{Var}(b_2) = \dfrac{\sigma^2}{14} = \dfrac{1}{14},
$$

because the $(3, 3)$ entry of $(X^{\mathsf T}X)^{-1}$ is $1/14$ and $\sigma^2 = 1$. The $95\%$ interval in each sample is

$$
b_2 \pm t_{2,\,0.975}\, s\sqrt{\dfrac{1}{14}}, \qquad s^2 = \dfrac{\mathrm{RSS}}{2}.
$$

Its coverage probability is $0.95$ when the normal model holds. Eight thousand independent samples should put the average of $b_2$ near $3$, the sample variance of $b_2$ near $1/14$, and the coverage near $0.95$. The experiment checks the code against those three theorems. It does not replace them.


In [2]:
# Sampling check: mean of b2, its variance, and 95 percent coverage.
import numpy as np
import scipy.stats as stats
rng = np.random.default_rng(0)
x = np.array([-2.0, -1, 0, 1, 2])
X = np.column_stack([np.ones(5), x, x ** 2])
gram_inv = np.linalg.inv(X.T @ X)
draws = 8000
y = 1 + 2 * x + 3 * x ** 2 + rng.normal(0.0, 1.0, size=(draws, 5))
beta = y @ X @ gram_inv
residuals = y - beta @ X.T
s = np.sqrt(np.sum(residuals ** 2, axis=1) / 2)
half = stats.t.ppf(0.975, 2) * s * np.sqrt(gram_inv[2, 2])
coverage = np.mean((beta[:, 2] - half <= 3) & (3 <= beta[:, 2] + half))
print("mean b2", beta[:, 2].mean(), "variance", beta[:, 2].var(ddof=1))
print("hand variance", gram_inv[2, 2])
print("coverage", coverage)
assert abs(beta[:, 2].mean() - 3) < 0.02
assert abs(beta[:, 2].var(ddof=1) - 1 / 14) < 0.01
assert abs(gram_inv[2, 2] - 1 / 14) < 1e-12
assert 0.935 < coverage < 0.965


mean b2 3.003101932611661 variance 0.07142844800719877
hand variance 0.07142857142857142
coverage 0.94525


## What a higher degree does to this table

Degree $3$ interpolates $y$ exactly, with leading coefficient $-5/6$. Degree $4$ has five coefficients and five points, so it also interpolates, and its coefficient of $x^4$ is zero because the cubic already fits. There is then no residual degree of freedom and no $s^2$. The quadratic is the last degree on this table that leaves a positive residual sum and a finite standard error. That is a property of these five numbers. A different sample from the same quadratic mean would not lie on a cubic.


In [3]:
# Degree 4 interpolates, and its leading coefficient is zero.
import numpy as np
x = np.array([-2.0, -1, 0, 1, 2])
y = np.array([10.0, 0, 1, 8, 16])
X = np.column_stack([x ** k for k in range(5)])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("degree 4", beta)
print("residual degrees of freedom", len(x) - X.shape[1])
assert np.allclose(beta[4], 0)
assert np.sum((y - X @ beta) ** 2) < 1e-8
assert len(x) - X.shape[1] == 0


degree 4 [ 1.00000000e+00  4.83333333e+00  3.00000000e+00 -8.33333333e-01
  3.23358125e-15]
residual degrees of freedom 0


## Pitfall

The $F$ statistic $126/5$ rejected a zero curvature on $2$ residual degrees of freedom, and the simulation covered the true curvature about $95\%$ of the time across $8{,}000$ draws. Those are different kinds of evidence. The single table is one draw, and its cubic interpolation shows how special that draw is. The coverage frequency is a property of the interval under repeated draws from the quadratic model. Quoting $R^2 = 83/88$ from five points as if it were the coverage probability confuses the two.


In [4]:
# One table produced one R-squared. Coverage was estimated from 8000 tables.
from fractions import Fraction
print("this table's R-squared", Fraction(83, 88))
print("interval's nominal coverage", Fraction(95, 100))
assert Fraction(83, 88) != Fraction(95, 100)


this table's R-squared 83/88
interval's nominal coverage 19/20


## Summary

- The study table fits $1 + 2x + 3x^2$, with $\mathrm{RSS} = 10$, $R^2 = 83/88$, and $F = 126/5$.
- At $x = 3$ the quadratic predicts $34$ and the interpolating cubic predicts $20$.
- Under normal errors of variance $1$, $\operatorname{Var}(b_2) = 1/14$, and the $95\%$ $t$ interval covers $3$ in about $95\%$ of repeated samples.
